# SignSense AI — MLP Training Notebook

**Model:** MLP landmark classifier (ASL A–Z + space/del/nothing = 29 classes)  
**Input:** 63-dim normalized MediaPipe landmark vector  
**Architecture:** Input(63) → Dense(512) → BN → ReLU → Dropout → Dense(256) → ... → Softmax(29)  
**Target accuracy:** > 95%  
**Runtime:** ~15 min on Colab T4 GPU

---
### Before you start
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your Kaggle API key (`kaggle.json`) when prompted, OR mount Google Drive with the dataset already downloaded
3. Run all cells top to bottom

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Create model output directory on Drive
import os
DRIVE_MODELS_DIR = '/content/drive/MyDrive/SignSense/models'
os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
print(f'Drive mounted. Models will be saved to: {DRIVE_MODELS_DIR}')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
# Colab already has TensorFlow + NumPy. Install the extras we need.
!pip install -q mediapipe==0.10.14 scikit-learn tqdm albumentations
print('Dependencies installed.')

In [ ]:
# ── Cell 3: Upload backend code to Colab ────────────────────────────────────
# Step 1: Run this PowerShell script LOCALLY to create the zip:
#   .\notebooks\create_colab_zip.ps1
#
# Step 2: Upload backend_colab.zip using the button below
#   (Files panel on the left → Upload, OR run the cell to get a file picker)

from google.colab import files
import os, sys, zipfile

BACKEND_PATH = '/content/backend'

if not os.path.exists(BACKEND_PATH):
    print('Upload backend_colab.zip when the file picker appears...')
    uploaded = files.upload()  # opens file picker
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('/content')
    print(f'Extracted {zip_name} to /content/')
else:
    print('backend/ already exists, skipping upload.')

# Add backend to Python path
sys.path.insert(0, BACKEND_PATH)

# Verify
from configs.training_config import ASL_CLASSES, NUM_CLASSES
print(f'Import OK — {NUM_CLASSES} classes: {ASL_CLASSES[:5]}...')

In [ ]:
# ── Cell 4: Download & preprocess ASL dataset ─────────────────────────────────
# Option A: Download from Kaggle
# Upload your kaggle.json first (Files panel → Upload)
import os

PROCESSED_NPY = '/content/backend/data/processed/ASL/landmarks_all.npy'

if not os.path.exists(PROCESSED_NPY):
    print('Downloading Kaggle ASL dataset...')
    # Upload kaggle.json via Files panel first, then run:
    !mkdir -p ~/.kaggle
    !cp /content/kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !pip install -q kaggle
    !kaggle datasets download -d grassknoted/asl-alphabet -p /content/asl_data --unzip

    # Create raw data directories and copy images
    !mkdir -p /content/backend/data/raw/ASL
    !cp -r /content/asl_data/asl_alphabet_train/* /content/backend/data/raw/ASL/

    # Run preprocessing pipeline
    print('Running preprocessing pipeline (this takes ~10-20 min)...')
    !python /content/backend/src/preprocess.py --all --augment --aug_factor 3
else:
    print(f'Preprocessed data already exists: {PROCESSED_NPY}')

# Verify
import numpy as np
X = np.load(PROCESSED_NPY)
y = np.load(PROCESSED_NPY.replace('landmarks_all', 'labels_all'))
print(f'X shape: {X.shape}  y shape: {y.shape}')

In [ ]:
# ── Cell 5: Verify GPU ────────────────────────────────────────────────────────
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('GPU memory growth enabled.')
else:
    print('WARNING: No GPU detected. Training will be slow on CPU.')

In [ ]:
# ── Cell 6: Train MLP ─────────────────────────────────────────────────────────
from pathlib import Path
from configs.training_config import MLPConfig
from src.train import train_mlp

cfg = MLPConfig()
cfg.save_dir = Path(DRIVE_MODELS_DIR)
cfg.log_dir  = Path('/content/logs/mlp')
cfg.mixed_precision = True  # Enable on T4 GPU for ~30% speedup

print('Config:')
print(f'  hidden_dims:    {cfg.hidden_dims}')
print(f'  dropout_rate:   {cfg.dropout_rate}')
print(f'  epochs:         {cfg.epochs}')
print(f'  batch_size:     {cfg.batch_size}')
print(f'  learning_rate:  {cfg.learning_rate}')
print(f'  mixed_precision:{cfg.mixed_precision}')
print()

model = train_mlp(cfg)
print('\nTraining complete!')

In [ ]:
# ── Cell 7: Evaluate ──────────────────────────────────────────────────────────
from src.evaluate import evaluate

# Temporarily point MODELS_DIR to Drive so evaluate() finds the model
import configs.training_config as tc
tc.MODELS_DIR = Path(DRIVE_MODELS_DIR)

results = evaluate('asl_mlp', split='test')
print(f"\nFinal test accuracy: {results['accuracy']*100:.1f}%")
print(f"Top-5 accuracy:      {results['top5']*100:.1f}%")

In [ ]:
# ── Cell 8: TensorBoard ───────────────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir /content/logs/mlp

In [ ]:
# ── Cell 9: Verify saved model ────────────────────────────────────────────────
import os
saved_files = os.listdir(DRIVE_MODELS_DIR)
print('Files saved to Drive:')
for f in saved_files:
    size_mb = os.path.getsize(os.path.join(DRIVE_MODELS_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

# Quick sanity check: load and run one prediction
import numpy as np
import tensorflow as tf
loaded = tf.keras.models.load_model(os.path.join(DRIVE_MODELS_DIR, 'asl_mlp.keras'))
dummy = np.zeros((1, 63), dtype=np.float32)
pred = loaded.predict(dummy, verbose=0)
print(f'\nSanity check — output shape: {pred.shape}  sum: {pred.sum():.4f} (should be ~1.0)')